# 📖 Notebook 4: Document Versioning and History

Every collaborative editor needs version history — the ability to see what changed, when, and by whom, and to restore previous versions. In this notebook, we explore how **snapshots** (compaction) and **operation logs** work together to provide efficient versioning.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why raw operation logs grow too large over time
- How snapshots (compaction) reduce storage and speed up loading
- How to implement version history and restore
- The trade-offs between storage, performance, and history granularity

## 🛠️ Setup

```bash
cd 06-system-designs/google-docs
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json
import time
import websockets
import asyncio

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "googledocs_demo",
    "user": "demo",
    "password": "demo"
}

WS_URL = "ws://localhost:8765"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def db_query(sql, params=None):
    conn = get_db()
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            return cur.fetchall()
    finally:
        conn.close()

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

## 🤔 The Problem: Operations Grow Forever

Every keystroke creates an operation. For an active document:

```
1 user typing at 5 keys/second × 3600 seconds = 18,000 operations per hour
10 users × 8 hours = 1,440,000 operations per day
```

When a new user opens the document, the server would need to:
1. Load **all** operations from the database
2. Replay them one by one to reconstruct the current text

For 1.4 million operations, that's slow and expensive!

**The solution**: periodically take a **snapshot** — save the full document text and discard (or archive) the old operations. New users only need the latest snapshot + any operations since then.

In [ ]:
# Let's visualize how operation counts grow vs snapshots

hours = list(range(1, 25))
ops_per_hour = 18000          # 1 user × 5 keystrokes/sec × 3600 sec
snapshot_every_hours = 2

# Without snapshots: every op ever written has to be replayed.
no_snapshot_ops = [ops_per_hour * h for h in hours]

# With snapshots: you replay only what has landed since the last one. We plot
# the WORST case — the moment just before the next snapshot fires — so the
# number is capped at one snapshot interval's worth of operations.
with_snapshot_ops = [min(ops_per_hour * h, ops_per_hour * snapshot_every_hours)
                     for h in hours]

print("📊 Operations to Replay When Loading Document")
print("=" * 60)
print(f"{'Hour':>4}  {'No Snapshots':>15}  {'Snapshot/2hr':>15}  Savings")
print("-" * 60)
for h in [1, 4, 8, 12, 24]:
    no_snap = no_snapshot_ops[h-1]
    with_snap = with_snapshot_ops[h-1]
    savings = ((no_snap - with_snap) / no_snap * 100) if no_snap > 0 else 0
    print(f"{h:>4}  {no_snap:>12,} ops  {with_snap:>12,} ops  {savings:.0f}%")

final_savings = (no_snapshot_ops[-1] - with_snapshot_ops[-1]) / no_snapshot_ops[-1]
print()
print(f"💡 After 24 hours, snapshots reduce ops to replay by {final_savings:.0%}!")
print("   This is why Google Docs periodically compacts operations.")
print("   Note the saving is not a fixed percentage — it grows without bound,")
print("   because the un-snapshotted cost grows while the snapshotted one is flat.")

# The whole point is that the snapshotted cost stops growing. Assert it, so a
# future edit to the model cannot quietly turn the lesson upside down.
assert with_snapshot_ops[-1] == with_snapshot_ops[len(with_snapshot_ops) // 2], (
    "ops-to-replay with snapshots is still growing with time — the model is wrong"
)
assert final_savings > 0.9, f"expected >90% savings at hour 24, got {final_savings:.0%}"

## 📸 What's a Snapshot?

A **snapshot** is a frozen copy of the complete document text at a specific point in time.

```
Operations:  [op1, op2, op3, ..., op50]  →  Snapshot v1: "Full document text..."
Operations:  [op51, op52, ..., op100]     →  Snapshot v2: "Updated full text..."
```

To load the document, you only need:
- The **latest snapshot** (full text)
- Plus any **operations after that snapshot** (usually very few)

Let's look at the snapshots in our database.

In [ ]:
# View existing snapshots
snapshots = db_query("""
    SELECT s.document_id, d.title, s.version, 
           LENGTH(s.content) as content_length,
           s.op_count, u.display_name as created_by,
           s.created_at
    FROM snapshots s
    JOIN documents d ON s.document_id = d.id
    JOIN users u ON s.created_by = u.id
    ORDER BY s.document_id, s.version
""")

print("📸 Existing Snapshots")
print("=" * 80)
print(f"{'Doc':>3}  {'Title':<35} {'Ver':>3}  {'Chars':>6}  {'Ops':>4}  {'By':<15}")
print("-" * 80)
for s in snapshots:
    title = s['title'][:33] + '..' if len(s['title']) > 33 else s['title']
    print(f"{s['document_id']:>3}  {title:<35} {s['version']:>3}  {s['content_length']:>6}  {s['op_count']:>4}  {s['created_by']:<15}")

print(f"\n💡 Version 0 = empty document (initial state)")
print(f"   Version 1 = first saved state (from seed data)")

In [ ]:
# View the content of snapshots for document 1
doc1_snapshots = db_query("""
    SELECT version, content, op_count, created_at
    FROM snapshots
    WHERE document_id = 1
    ORDER BY version
""")

print("📄 Document 1: 'Meeting Notes — Q1 Planning'")
print("=" * 60)
for snap in doc1_snapshots:
    print(f"\n📸 Version {snap['version']} ({snap['op_count']} ops compacted):")
    print(f"   Created: {snap['created_at']}")
    if snap['content']:
        for line in snap['content'].split('\n'):
            print(f"   │ {line}")
    else:
        print(f"   │ (empty document)")

## 🔄 Loading a Document: Snapshot + Replay

When a user opens a document, the server:
1. Loads the **latest snapshot** from the `snapshots` table
2. Loads any **operations after that snapshot** from the `operations` table
3. Replays those operations on top of the snapshot

This is much faster than replaying ALL operations from the beginning.

In [ ]:
def load_document_from_db(doc_id):
    """Load a document using snapshot + replay.

    This is exactly what the doc server does when loading a document.
    Returns (text, version, ops_replayed).
    """
    # Step 1: Get the latest snapshot
    snapshots = db_query(
        "SELECT version, content FROM snapshots "
        "WHERE document_id = %s ORDER BY version DESC LIMIT 1",
        (doc_id,)
    )

    if snapshots:
        text = snapshots[0]["content"]
        version = snapshots[0]["version"]
        print(f"  📸 Loaded snapshot v{version} ({len(text)} chars)")
    else:
        text = ""
        version = 0
        print(f"  📸 No snapshot found, starting empty")

    # Step 2: Get operations since the snapshot
    ops = db_query(
        "SELECT op_type, position, content, length FROM operations "
        "WHERE document_id = %s AND version >= %s ORDER BY id",
        (doc_id, version)
    )

    print(f"  🔄 Found {len(ops)} operations to replay")

    # Step 3: Replay operations
    for op in ops:
        pos = op["position"]
        if op["op_type"] == "insert":
            text = text[:pos] + (op["content"] or "") + text[pos:]
        elif op["op_type"] == "delete":
            length = op["length"] or 1
            text = text[:pos] + text[pos + length:]

    print(f"  ✅ Document loaded: {len(text)} chars")
    return text, version, len(ops)

print("Loading Document 1:")
text, version, replayed = load_document_from_db(1)
print(f"\n📄 Content preview:")
print(text[:200])

# The shortcut is only a shortcut if it gives the same answer as the long way
# round. Replay EVERY operation from an empty document and compare.
all_ops = db_query(
    "SELECT op_type, position, content, length FROM operations "
    "WHERE document_id = 1 ORDER BY id"
)
full = ""
for op in all_ops:
    pos = op["position"]
    if op["op_type"] == "insert":
        full = full[:pos] + (op["content"] or "") + full[pos:]
    else:
        full = full[:pos] + full[pos + (op["length"] or 1):]

print()
print(f"  snapshot + replay : {len(text):>4} chars, {replayed:>3} ops replayed")
print(f"  full replay       : {len(full):>4} chars, {len(all_ops):>3} ops replayed")

assert text == full, (
    "snapshot+replay and full replay disagree — either the snapshot is stale or "
    "the operations log no longer rebuilds the document"
)
assert replayed <= len(all_ops), "the snapshot made loading *more* expensive"
print("  ✅ identical text — the snapshot is a faithful compaction of the ops before it")
print()
print("⚠️  That equality holds because document 1 has only ever been edited.")
print("   A version *restore* (below) rewrites the text without emitting ops, so")
print("   afterwards a full replay of doc 2 no longer reproduces it — the snapshot")
print("   chain, not the op log, is the source of truth for history.")

## 💾 Creating Snapshots via the Server

Let's connect to the doc server, make some edits, and create a snapshot.

In [ ]:
async def create_edits_and_snapshot(doc_id, user_id=1):
    """Connect, make edits, and save a snapshot."""
    async with websockets.connect(WS_URL) as ws:
        # Connect
        await ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        await ws.recv()  # connected
        
        # Join document
        await ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        doc_state = json.loads(await ws.recv())  # doc_state
        await ws.recv()  # presence_list
        
        doc_text = doc_state["text"]
        print(f"📄 Joined doc {doc_id} (version {doc_state['version']})")
        print(f"   Current length: {len(doc_text)} chars")
        
        # Make some edits
        edits = [
            "\n\n--- New Section ---",
            "\nThis text was added to demonstrate versioning.",
            "\nTimestamp: " + time.strftime("%Y-%m-%d %H:%M:%S"),
        ]
        
        for edit_text in edits:
            pos = len(doc_text)
            await ws.send(json.dumps({
                "type": "edit",
                "document_id": doc_id,
                "op_type": "insert",
                "position": pos,
                "content": edit_text,
            }))
            await ws.recv()  # ack
            doc_text += edit_text
            print(f"   ✏️  Inserted {len(edit_text)} chars")
        
        # Save a snapshot
        await ws.send(json.dumps({
            "type": "save_snapshot",
            "document_id": doc_id,
        }))
        resp = json.loads(await ws.recv())
        print(f"   📸 Snapshot saved: version {resp.get('version', '?')}")

        resp["expected_text"] = doc_text
        return resp

before_version = db_query(
    "SELECT COALESCE(MAX(version), -1) AS v FROM snapshots WHERE document_id = 2"
)[0]["v"]

result = await create_edits_and_snapshot(2)  # Use document 2

# A snapshot is only useful if it froze the text the editors actually had.
saved = db_query(
    "SELECT content FROM snapshots WHERE document_id = 2 AND version = %s",
    (result["version"],)
)
assert saved, f"no snapshot row was written for version {result['version']}"
assert result["version"] > before_version, "save_snapshot did not advance the version"
assert saved[0]["content"] == result["expected_text"], (
    "the snapshot content does not match the document the client just built — "
    "compaction lost or reordered an operation"
)
print(f"\n   ✅ Snapshot v{result['version']} matches the client's text exactly "
      f"({len(result['expected_text'])} chars)")

In [ ]:
# Check what snapshots exist now
doc2_snapshots = db_query("""
    SELECT version, LENGTH(content) as chars, op_count, created_at
    FROM snapshots
    WHERE document_id = 2
    ORDER BY version
""")

print("📸 Snapshots for Document 2:")
print(f"{'Version':>7}  {'Chars':>6}  {'Ops Compacted':>14}  Created")
print("-" * 60)
for s in doc2_snapshots:
    print(f"{s['version']:>7}  {s['chars']:>6}  {s['op_count']:>14}  {s['created_at']}")

## 🕐 Version History and Restore

With snapshots, we can implement **version history** — showing users what the document looked like at each saved point, and letting them restore to a previous version.

In [ ]:
async def get_version_history(doc_id, user_id=1):
    """Fetch version history from the server."""
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        await ws.recv()
        
        await ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        await ws.recv()  # doc_state
        await ws.recv()  # presence_list
        
        await ws.send(json.dumps({
            "type": "get_history",
            "document_id": doc_id,
        }))
        resp = json.loads(await ws.recv())
        return resp

history = await get_version_history(2)

print("🕐 Version History for Document 2:")
print("=" * 60)
for v in history.get("versions", []):
    content_preview = v["content"][:80] + "..." if len(v["content"]) > 80 else v["content"]
    print(f"\n  📸 Version {v['version']}:")
    print(f"     Created by: {v['created_by']}")
    print(f"     Created at: {v['created_at']}")
    print(f"     Ops compacted: {v['op_count']}")
    print(f"     Content: {content_preview}")

In [ ]:
# Demonstrate version restore
async def restore_version(doc_id, version, user_id=1):
    """Restore a document to a previous version."""
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        await ws.recv()

        await ws.send(json.dumps({"type": "join_doc", "document_id": doc_id}))
        state = json.loads(await ws.recv())  # doc_state
        await ws.recv()  # presence_list

        print(f"Current document ({len(state['text'])} chars):")
        print(f"  '{state['text'][:100]}...'")
        print()

        await ws.send(json.dumps({
            "type": "restore_version",
            "document_id": doc_id,
            "version": version,
        }))

        # Read the response(s)
        restored_text = None
        resp = json.loads(await ws.recv())
        # May receive doc_state broadcast first
        if resp["type"] == "doc_state":
            restored_text = resp["text"]
            print(f"Restored to version {version} ({len(resp['text'])} chars):")
            print(f"  '{resp['text'][:100]}...' " if len(resp['text']) > 100 else f"  '{resp['text']}'")
            resp = json.loads(await ws.recv())

        if resp["type"] == "version_restored":
            print(f"\n✅ Restored version {resp['restored_version']} → new version {resp['new_version']}")
            return resp, restored_text
        return resp, restored_text

print("Restoring Document 2 to version 1 (original content):")
print("=" * 60)
target_version = 1
target_text = db_query(
    "SELECT content FROM snapshots WHERE document_id = 2 AND version = %s",
    (target_version,)
)[0]["content"]

resp, restored_text = await restore_version(2, target_version)

# "Restore" has to mean the text is byte-for-byte the old version, and it has to
# be recorded as a NEW version rather than rewriting history.
assert restored_text == target_text, (
    "the restored document is not what version 1 contained — restore is lossy"
)
new_snapshot = db_query(
    "SELECT content, version FROM snapshots WHERE document_id = 2 AND version = %s",
    (resp["new_version"],)
)
assert new_snapshot, "restore did not write a new snapshot"
assert new_snapshot[0]["content"] == target_text
assert resp["new_version"] > target_version, (
    "restore overwrote history instead of appending a new version"
)
print(f"   Version {target_version} is still in the history, and v{resp['new_version']} "
      f"now holds the same {len(target_text)} characters.")

## 🔍 Diffing Between Versions

Google Docs' "See new changes" and "Show revision history" features compare
two snapshots and highlight what was **added** and **removed**. We can build a
simple version of that using Python's built-in `difflib` — no extra libraries.

In a real system this is usually done **client-side** on the two text strings
returned by the server, so the server stays a simple data store.

In [ ]:
# Compare two snapshots and show what changed
import difflib

# Use document 1 (Meeting Notes) which has multi-line content.
snaps = db_query("""
    SELECT version, content FROM snapshots
    WHERE document_id = 1 ORDER BY version
""")

if len(snaps) >= 2:
    # The two most recent versions — diffing against the empty v0 would just
    # print the whole document as one giant addition, which teaches nothing.
    older, newer = snaps[-2], snaps[-1]
    print(f"Diff: doc 1, v{older['version']} -> v{newer['version']}")
    print("=" * 60)

    old_lines = older["content"].splitlines(keepends=True)
    new_lines = newer["content"].splitlines(keepends=True)

    diff = list(difflib.unified_diff(
        old_lines, new_lines,
        fromfile=f"v{older['version']}",
        tofile=f"v{newer['version']}",
        lineterm="",
    ))
    for line in diff:
        # Strip trailing newline so each diff line prints on one line
        clean = line.rstrip("\n")
        if clean.startswith("+") and not clean.startswith("+++"):
            print(f"\033[32m{clean}\033[0m")  # green for additions
        elif clean.startswith("-") and not clean.startswith("---"):
            print(f"\033[31m{clean}\033[0m")  # red for removals
        else:
            print(clean)

    print()
    print("This is how 'Show changes since last view' works in most editors.")
    print("Under the hood: store snapshots, compute diff on demand.")

    # A diff that comes back empty for two different texts means the snapshots
    # are duplicates — which is exactly what an over-eager compaction produces.
    if older["content"] != newer["content"]:
        assert diff, "two different snapshots produced an empty diff"
        added = sum(1 for l in diff if l.startswith("+") and not l.startswith("+++"))
        removed = sum(1 for l in diff if l.startswith("-") and not l.startswith("---"))
        print(f"\n{added} line(s) added, {removed} line(s) removed between these versions.")
    else:
        print("\n(These two snapshots are identical — run Notebook 3 to add some edits.)")
else:
    print("Need at least 2 snapshots to diff.")

## 🗜️ Compaction Strategy

In a production system like Google Docs, compaction (snapshotting) happens strategically:

| Trigger | When | Why |
|---------|------|-----|
| **Operation count** | Every N ops (e.g., 50) | Prevents unbounded growth |
| **Idle document** | When last editor disconnects | Safe — no concurrent ops |
| **Manual save** | User clicks "Save" | User-initiated checkpoint |
| **Scheduled** | Periodic background job | Catches long-running sessions |

Our server auto-snapshots every **50 operations** and when the **last user disconnects**.

In [ ]:
# Let's look at what compaction actually costs and buys

# Count operations vs snapshot sizes
op_stats = db_query("""
    SELECT document_id, COUNT(*) as op_count,
           SUM(LENGTH(COALESCE(content, ''))) as total_content_bytes
    FROM operations
    GROUP BY document_id
""")

snap_stats = db_query("""
    SELECT document_id, COUNT(*) as snap_count,
           SUM(LENGTH(content)) as total_snap_bytes,
           MAX(version) as latest_version
    FROM snapshots
    GROUP BY document_id
""")

print("📊 Storage: Operations vs Snapshots")
print("=" * 60)
print(f"{'Doc':>3}  {'Ops':>5}  {'Op Bytes':>10}  {'Snaps':>5}  {'Snap Bytes':>10}")
print("-" * 60)

totals = {"ops": 0, "op_bytes": 0, "snaps": 0, "snap_bytes": 0}
for op in op_stats:
    doc_id = op['document_id']
    snap = next((s for s in snap_stats if s['document_id'] == doc_id), None)
    snap_count = snap['snap_count'] if snap else 0
    snap_bytes = (snap['total_snap_bytes'] or 0) if snap else 0
    print(f"{doc_id:>3}  {op['op_count']:>5}  {op['total_content_bytes'] or 0:>10}  {snap_count:>5}  {snap_bytes:>10}")
    totals["ops"] += op["op_count"]
    totals["op_bytes"] += op["total_content_bytes"] or 0
    totals["snaps"] += snap_count
    totals["snap_bytes"] += snap_bytes

print()
print("💡 Read that table carefully: snapshots are NOT a storage win.")
print("   Every snapshot stores the WHOLE document again, so the snapshot bytes")
print(f"   ({totals['snap_bytes']:,}) climb with each compaction while the op log")
print(f"   ({totals['op_bytes']:,} bytes of content) only records the deltas.")
print()
print("   What compaction buys is LOAD TIME — replaying 40 ops instead of 40,000.")
print("   The storage win only arrives afterwards, when the ops the snapshot")
print("   subsumes are archived to cold storage or deleted; keep them and you are")
print("   paying for both. That retention choice is the real design decision:")
print("     keep all ops    → full audit trail + character-level undo, more storage")
print("     drop old ops    → history granularity limited to snapshot boundaries")

assert totals["snaps"] > 0 and totals["ops"] > 0, "no data to compare"

## 🏗️ Production Considerations

### Google Docs' Approach

```
Document Service
     │
     ├─── On every edit: append to operations log (fast)
     │
     ├─── Every 50 ops: auto-snapshot (background)
     │
     ├─── On last disconnect: final snapshot + compaction
     │
     └─── Version metadata: stored in Document Metadata DB
          (which snapshot version is "current")
```

### Key Design Decisions

| Decision | Trade-off |
|----------|----------|
| Snapshot frequency | More often = faster loads, but more storage + CPU |
| Keep old operations | Yes = full audit trail, but storage grows |
| Version retention | Keep all = unlimited undo, but more storage |
| Compaction timing | During idle = safe; during editing = risky |

### Billions of Documents

With billions of documents at ~50KB each:
- Raw storage: **50 TB** just for current document text
- With operations + versions: could be **10-100× more**
- Compaction is **essential** to keep storage manageable

## 🧹 Cleanup

In [ ]:
print("🧹 No cleanup needed — snapshots are part of the demo data.")
print("   To fully reset, run: docker compose down -v && docker compose up -d")

## 📚 Summary

### Key Takeaways

1. **Operations grow unbounded** — every keystroke is logged, leading to millions of ops per document
2. **Snapshots (compaction)** save the full document text periodically, enabling fast loading
3. **Loading = latest snapshot + replay** — only replay ops since the last snapshot
4. **Version history** = a chain of snapshots, each with the full document at that point
5. **Restore** = create a new snapshot with the content from an old version

### What This Toy Versioning Does Not Do

- **No per-keystroke history.** Google Docs can show you the document at any
  moment; we can only show it at snapshot boundaries. Getting the in-between
  states back means keeping the op log forever and replaying it.
- **Snapshots are not free.** Each one stores the whole document again, so
  compaction *costs* storage until the ops it subsumes are archived.
- **Restore is not an operation.** It rewrites the text and starts a new
  version, so replaying the op log stops reproducing the document afterwards
  (and any client mid-edit has its revision number invalidated).
- **No retention policy.** Real systems expire old versions, dedupe unchanged
  snapshots, and store deltas between them rather than full copies.

### For System Design Interviews

- Always mention **compaction/snapshots** when discussing operation logs
- Explain the **trade-off**: snapshot frequency vs storage vs load time
- Note that compaction should happen when the document is **idle** (no active editors)
- Version history is built **on top of** snapshots — not a separate system

### What We've Covered in This Lab

| Notebook | Topic | Key Concept |
|----------|-------|-------------|
| 1 | Operational Transformation | Transform concurrent ops to preserve intent |
| 2 | CRDTs | Order-independent operations, no central server |
| 3 | Real-Time Collaboration | WebSockets, presence, scaling with consistent hashing |
| 4 | Versioning & History | Snapshots, compaction, version restore |

Together, these form the core of a Google Docs-style collaborative document editor. 🎉